In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load csv results
hrv_results_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv')
pupil_results_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv')
duration_results_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv')

# combine with suffixes
combined_results = pd.concat([
    hrv_results_df.set_index('Participant').add_suffix('_HRV_SDNN'),
    pupil_results_df.set_index('Participant').add_suffix('_Pupil_STD'),
    duration_results_df.set_index('Participant').add_suffix('_Duration_STD'),
], axis=1)

# correlation matrix
correlation_matrix = combined_results.corr(method='spearman')

# plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of HRV SDNN, Pupil Dilation STD, and Duration STD')
plt.show()
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load hrv + pupil
hrv_results_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv')
pupil_results_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv')

# indexed lookup
hrv_idx = hrv_results_df.set_index('Participant')
pupil_idx = pupil_results_df.set_index('Participant')

for pid in hrv_idx.index:
    for s in [1, 2, 3]:
        col = f'Session {s:02d}'
        print(f"Participant {pid}, Session {s}, HRV SDNN: {hrv_idx.loc[pid, col]:.2f}")
        print(f"Participant {pid}, Session {s}, Pupil Dilation STD: {pupil_idx.loc[pid, col]:.2f}")

# combine with suffixes
combined_results = pd.concat([
    hrv_results_df.set_index('Participant').add_suffix('_HRV_SDNN'),
    pupil_results_df.set_index('Participant').add_suffix('_Pupil_STD'),
], axis=1)

# correlation matrix
correlation_matrix = combined_results.corr(method='spearman')

# plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of HRV SDNN and Pupil Dilation STD')
plt.show()
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT = '../data/group_results'

# load data
hrv_df = pd.read_csv(f'{OUTPUT}/HRV_SDNN.csv').set_index('Participant')
pupil_df = pd.read_csv(f'{OUTPUT}/Pupil_Dilation_STD.csv').set_index('Participant')
duration_df = pd.read_csv(f'{OUTPUT}/Psychometric_Test_Duration_STD.csv').set_index('Participant')

# combine all
combined = pd.concat([
    hrv_df.add_suffix('_HRV_SDNN'),
    pupil_df.add_suffix('_Pupil_STD'),
    duration_df.add_suffix('_Duration_STD'),
], axis=1)

cols = combined.columns
n = len(cols)

# correlation with p-values
r_matrix = np.zeros((n, n))
p_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        r, p = stats.pearsonr(combined.iloc[:, i], combined.iloc[:, j])
        r_matrix[i, j] = r
        p_matrix[i, j] = p

r_df = pd.DataFrame(r_matrix, index=cols, columns=cols)
p_df = pd.DataFrame(p_matrix, index=cols, columns=cols)

# annotation with significance stars
annot = np.empty_like(r_matrix, dtype=object)
for i in range(n):
    for j in range(n):
        stars = "***" if p_df.iloc[i, j] < 0.001 else "**" if p_df.iloc[i, j] < 0.01 else "*" if p_df.iloc[i, j] < 0.05 else ""
        annot[i, j] = f"{r_df.iloc[i, j]:.2f}{stars}"

# plot heatmap
plt.figure(figsize=(14, 11))
sns.heatmap(r_df, annot=annot, fmt='', cmap='coolwarm', vmin=-1, vmax=1, center=0)
plt.title('Correlation Matrix with Significance (* p<.05, ** p<.01, *** p<.001)')
plt.tight_layout()
plt.show()
plt.close()